<a href="https://colab.research.google.com/github/orionrjsun-collab/BUS4-118s/blob/main/Coding_Exercise_ML_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: Predict house prices based on square footage and location.

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

'''
# Generate sample data
data = {
'square_footage': [1500, 2000, 1800, 2500, 2200, 1700, 3000, 1900, 2100, 2600],
'location': ['Downtown', 'Suburb', 'Downtown', 'Rural', 'Suburb', 'Downtown',
'Rural', 'Suburb', 'Downtown', 'Rural'],
'price': [300000, 350000, 320000, 280000, 360000, 310000, 400000, 340000,
330000, 290000]
}
df = pd.DataFrame(data)
'''
#Data Source: https://www.kaggle.com/datasets/marcopale/housing
df = pd.read_csv('AmesHousing.csv')
df = df[['Gr Liv Area', 'Neighborhood', 'SalePrice']]
df.columns = ['square_footage', 'location', 'price']
print(df.columns)

# Features and target
X = df[['square_footage', 'location']]
y = df['price']

# Preprocessing: One-hot encode the location column
preprocessor = ColumnTransformer(
transformers=[('location', OneHotEncoder(sparse_output=False), ['location'])], remainder='passthrough')

# Create pipeline with preprocessing and model
model = Pipeline(steps=[('preprocessor', preprocessor),('regressor', LinearRegression())])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)

# Train model
model.fit(X_train, y_train)

# Make prediction for a new house: 2000 sq ft in Downtown
new_house = pd.DataFrame({'square_footage': [2000], 'location': ['OldTown']})
predicted_price = model.predict(new_house)
print(f"Predicted price for a 2000 sq ft house in Downtown: ${predicted_price[0]:,.2f}")

# Display model coefficients
feature_names = (model.named_steps['preprocessor'].named_transformers_['location'].get_feature_names_out(['location'])).tolist() +['square_footage']
coefficients = model.named_steps['regressor'].coef_
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
  print(f"{feature}: {coef:.2f}")

Index(['square_footage', 'location', 'price'], dtype='object')
Predicted price for a 2000 sq ft house in Downtown: $165,143.95

Model Coefficients:
location_Blmngtn: 16052.73
location_Blueste: -31746.42
location_BrDale: -52182.26
location_BrkSide: -40070.18
location_ClearCr: 5503.05
location_CollgCr: 16539.84
location_Crawfor: -990.87
location_Edwards: -40455.07
location_Gilbert: -4677.04
location_Greens: 33147.05
location_GrnHill: 101784.11
location_IDOTRR: -60047.06
location_Landmrk: -35241.85
location_MeadowV: -56630.35
location_Mitchel: -13719.61
location_NAmes: -24124.91
location_NPkVill: -25591.89
location_NWAmes: -13420.43
location_NoRidge: 68932.19
location_NridgHt: 97111.37
location_OldTown: -58847.57
location_SWISU: -62908.94
location_Sawyer: -26274.99
location_SawyerW: -7718.92
location_Somerst: 33161.63
location_StoneBr: 105294.86
location_Timber: 39505.39
location_Veenker: 37616.15
square_footage: 76.10


Part 2: Predict Customer Churn

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Generate sample customer data
'''
data = {
'age': [25, 34, 45, 28, 52, 36, 41, 29, 47, 33],
'monthly_usage_hours': [10, 50, 20, 15, 60, 30, 25, 12, 55, 40],
'purchase_amount': [100, 250, 150, 80, 300, 200, 175, 90, 280, 220],
'customer_service_calls': [5, 2, 8, 6, 1, 3, 7, 4, 0, 2],
'region': ['North', 'South', 'West', 'East', 'South', 'North', 'West', 'East','South', 'North'],
'churn': [1, 0, 1, 1, 0, 0, 1, 1, 0, 0] # 1 = churned, 0 = not churned
}
df = pd.DataFrame(data)
'''

#Used ChatGPT to generate: https://chatgpt.com/share/698d7737-dca0-800d-b00e-1afafaf1a9af
df = pd.read_csv('/content/customer_churn_data.csv')


# Features and target
X = df[['age', 'monthly_usage_hours', 'purchase_amount', 'customer_service_calls','region']]
y = df['churn']

# Preprocessing: Scale numerical features and one-hot encode categorical features
preprocessor = ColumnTransformer(
transformers=[('num', StandardScaler(), ['age', 'monthly_usage_hours', 'purchase_amount','customer_service_calls']),('cat', OneHotEncoder(sparse_output=False), ['region'])])

# Create pipeline with preprocessing and model
model = Pipeline(steps=[('preprocessor', preprocessor),('classifier', LogisticRegression(random_state=42))])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)

# Train model
model.fit(X_train, y_train)

# Predict churn probability for a new customer
new_customer = pd.DataFrame({
'age': [35],
'monthly_usage_hours': [20],
'purchase_amount': [150],
'customer_service_calls': [5],
'region': ['West']
})
churn_probability = model.predict_proba(new_customer)[0][1] # Probability of churn (class 1)

# Classify based on threshold (0.5)
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0
print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

# Display model coefficients
feature_names = (model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(['region'])).tolist() + ['age','monthly_usage_hours', 'purchase_amount', 'customer_service_calls']
coefficients = model.named_steps['classifier'].coef_[0]
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
  print(f"{feature}: {coef:.2f}")

Churn Probability for new customer: 0.22
Churn Prediction (1 = churn, 0 = no churn): 0

Model Coefficients:
region_East: -0.35
region_North: -0.03
region_South: -0.16
region_West: 0.15
age: 0.50
monthly_usage_hours: 0.00
purchase_amount: -0.96
customer_service_calls: 0.46


In [20]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

'''
# Generate sample customer data
data = {
'annual_spending': [500, 1200, 300, 1500, 800, 200, 1000, 600, 1300, 400],
'purchase_frequency': [5, 12, 3, 15, 8, 2, 10, 6, 13, 4],
'age': [25, 34, 45, 28, 52, 36, 41, 29, 47, 33],
'region': ['North', 'South', 'West', 'East', 'South', 'North', 'West', 'East',
'South', 'North']
}
df = pd.DataFrame(data)
'''
#Used ChatGPT to generate: https://chatgpt.com/share/698d7737-dca0-800d-b00e-1afafaf1a9af
df = pd.read_csv('/content/customer_spending_data.csv')

# Preprocess data: Select numerical features and scale them
features = ['annual_spending', 'purchase_frequency', 'age']
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Determine optimal number of clusters using elbow method
inertia = []
K = range(1, 6)
for k in K:
  kmeans = KMeans(n_clusters=k, random_state=42)
  kmeans.fit(X_scaled)
  inertia.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(8, 5))
plt.plot(K, inertia, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.savefig('elbow_plot.png')
plt.close()

# Apply K-Means with optimal K (e.g., 3 based on elbow method)
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Analyze clusters
cluster_summary = df.groupby('cluster')[features].mean().round(2)
print("Cluster Characteristics:")
print(cluster_summary)

# Example of targeted strategies
for cluster in range(optimal_k):
  print(f"\nCluster {cluster} Strategy:")
  if cluster_summary.loc[cluster, 'annual_spending'] > 1000:
    print("High-spending customers: Offer exclusive promotions or loyaltyrewards.")
  elif cluster_summary.loc[cluster, 'purchase_frequency'] > 10:
    print("Frequent buyers: Provide bulk discounts or subscription plans.")
  else:
    print("Low-engagement customers: Send personalized re-engagement campaigns.")

# Save cluster assignments to CSV
df.to_csv('customer_segments.csv', index=False)

Cluster Characteristics:
         annual_spending  purchase_frequency    age
cluster                                            
0                 721.61                6.63  56.75
1                 739.48                7.31  27.77
2                1180.04                9.85  44.57

Cluster 0 Strategy:
Low-engagement customers: Send personalized re-engagement campaigns.

Cluster 1 Strategy:
Low-engagement customers: Send personalized re-engagement campaigns.

Cluster 2 Strategy:
High-spending customers: Offer exclusive promotions or loyaltyrewards.
